# 02: Data Conditioning

Type conversions, defaults imputation (config `imputation`), then remaining numeric NaN -> 0.

**Input:** data/01_assembled.parquet
**Output:** data/02_conditioned.parquet

In [ ]:
# Parameters (injected by papermill)
config_path = "config/car_coll/v1"

In [ ]:
import pandas as pd
import numpy as np
import yaml
import os
import sys
import gc
from pathlib import Path

sys.path.insert(0, str(Path.cwd() / 'lib'))
from utils import setup_notebook_environment

print("Stage 02: Data conditioning")

project_root = setup_notebook_environment()
print(f"\nProject root: {project_root}")

In [ ]:
# Load config
config_file = f"{config_path}/config.yaml"
with open(config_file, 'r') as f:
    cfg = yaml.safe_load(f)

pc_num = open('current.pc').read().strip()
pc_id = f'PC{pc_num}'
output_base = cfg['machines'][pc_id]['paths']['output_path']
print(f"\nExperiment: {cfg['experiment']['name']}")
print(f"Target: {cfg['experiment']['target']}")

In [ ]:
# Load 01 output
checkpoint_01 = f"{output_base}/data/01_assembled.parquet"
print(f"\nLoading checkpoint: {checkpoint_01}")

data = pd.read_parquet(checkpoint_01)
print(f"  Shape: {data.shape}")
print(f"  Memory: {data.memory_usage(deep=True).sum() / 1e9:.2f} GB")

In [ ]:
# Check data types
print(f"\nData type distribution:")
print(data.dtypes.value_counts())

In [ ]:
# Check for missing values
missing_counts = data.isnull().sum()
missing_cols = missing_counts[missing_counts > 0]

if len(missing_cols) > 0:
    print(f"\nMissing values found in {len(missing_cols)} columns:")
    for col, count in missing_cols.head(10).items():
        pct = (count / len(data)) * 100
        print(f"  {col}: {count:,} ({pct:.2f}%)")
    if len(missing_cols) > 10:
        print(f"  ... and {len(missing_cols) - 10} more")
else:
    print(f"\nNo missing values found")

In [ ]:
# Type conversions, defaults imputation (config 'imputation'), then numeric NaN -> 0.
# Same function as 08 scoring, so both condition the data the same way.
from preprocessing import condition_data
data = condition_data(data, cfg, type_conversions_path=f"{config_path}/type_conversions.csv", copy=False)
print(f"  Conditioned: {data.shape}")

In [ ]:
# Verify target variable exists
target = cfg['experiment']['target']
if target in data.columns:
    print(f"\nTarget variable '{target}' verified")
    print(f"  Non-null count: {data[target].notnull().sum():,}")
    print(f"  Mean: {data[target].mean():.4f}")
    print(f"  Median: {data[target].median():.4f}")
    print(f"  Min: {data[target].min():.4f}")
    print(f"  Max: {data[target].max():.4f}")
else:
    print(f"\nERROR: Target variable '{target}' not found!")
    print(f"  Available columns: {data.columns.tolist()[:20]}")

In [ ]:
from utils import validate_critical_columns
validate_critical_columns(data, cfg, 'Stage 02 Output')

# Save conditioned checkpoint
checkpoint_02 = f"{output_base}/data/02_conditioned.parquet"
print(f"\nSaving checkpoint: {checkpoint_02}")

from utils import save_parquet
from utils import downcast_frame
data = downcast_frame(data)  # floats -> float32, ints -> smallest
save_parquet(data, checkpoint_02)
file_size = os.path.getsize(checkpoint_02) / 1e9

print(f"  Saved: {file_size:.2f} GB")
print(f"  Shape: {data.shape}")

In [ ]:
print(f"\nCleaning memory...")
del data
gc.collect()
print(f"  Memory cleaned")

In [ ]:
print("Stage 02 done")